---
### Environment, Ingestion & Strict Train-Test Split

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAW_DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw = pd.read_csv(RAW_DATA_PATH)

# Clean TotalCharges blank anomalies as established in Milestone 1
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)

# Binary target
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# Separate features from target and identifiers
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

# Strict stratified split before ANY feature engineering or statistics calculation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"X_test shape: {X_test.shape}, y_test distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (5634, 19), y_train distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
X_test shape: (1409, 19), y_test distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


> That output confirms three critical properties before we do any feature transformation or model fitting:
* **Exact 80/20 Partition Integrity:** The dataset ($7,043$ total records) split cleanly into $5,634$ training rows and $1,409$ holdout test rows, while stripping out customerID and Churn to leave $19$ raw predictive features.
* **Stratification Worked (stratify=y):** Both partitions preserve the exact class distribution—$73.46\%$ retained (0) and $26.54\%$ churned (1). Because churn is moderately imbalanced (~1:3 ratio), a random unstratified split could have skewed the test set's churn rate and distorted our threshold tuning later.
* **Evaluation Metric Choice is Validated:** Having a $26.5\%$ positive class rate confirms why raw accuracy is a poor metric here—a naive model predicting "No Churn" for everyone would automatically score $73.46\%$ accuracy while missing $100\%$ of churners. This justifies optimizing for PR-AUC and calibrating a custom cost threshold in Milestone 3.

> Note: 

PR-AUC stands for Precision-Recall Area Under the Curve. It summarizes a classification model's performance across all possible probability decision thresholds ($0.0$ to $1.0$), specifically focusing on how well the model identifies the minority (positive) class—in our case, customers who churn (Churn = 1).
* Recall (y-axis) — "Catch Rate": Out of all customers who actually churned, what fraction did the model flag?$$\text{Recall} = \frac{TP}{TP + FN}$$
* Precision (x-axis) — "Alert Quality": Out of all customers the model flagged as likely to churn, what fraction actually churned?$$\text{Precision} = \frac{TP}{TP + FP}$$

In our churn prediction problem, **`1` = Churned (Positive)** and **`0` = Retained (Negative)**. Every customer prediction falls into one of these four confusion matrix outcomes:

| Outcome | Actual (`y_test`) | Predicted (`y_pred`) | What It Means in SaaS Churn | Business & Financial Consequence |
| --- | --- | --- | --- | --- |
| **TP (True Positive)** | **1** (Churned) | **1** (Predicted Churn) | **Caught a churner.** The model flagged an at-risk account before they left. | **Win:** Customer Success intervenes and saves the account's Annual Contract Value (ACV). |
| **FN (False Negative)** | **1** (Churned) | **0** (Predicted Stay) | **Missed a churner.** The model thought they were safe, and they canceled silently. | **Severe Loss ($500):** You lose the customer's entire lifetime value (LTV) with zero chance to intervene. |
| **FP (False Positive)** | **0** (Retained) | **1** (Predicted Churn) | **False alarm.** The model flagged a loyal customer who wasn't going to leave. | **Minor Cost ($50):** You spend unnecessary retention budget (a discount coupon or CS call) on a happy customer. |
| **TN (True Negative)** | **0** (Retained) | **0** (Predicted Stay) | **Correctly ignored.** A loyal customer stayed, and the model left them alone. | **Neutral ($0):** Normal operations; zero wasted outreach effort. |


---
### Feature Engineering Function (Leakage-Safe)

We package domain features into an explicit function applied uniformly to train and test partitions without computing global cross-record aggregates:

**Note:** $1\mathcal{e}^{-5} = 1 \times 10^{-5} = 0.00001 $ used to prevents zero-division error.

In [2]:
def engineer_saas_features(df_input: pd.DataFrame) -> pd.DataFrame:
    df_feat = df_input.copy()

    # 1. ChargeRatio: Captures billing consistency vs lifetime accrual
    # Small epsilon prevents zero-division for month-0 users
    df_feat["ChargeRatio"] = df_feat["TotalCharges"] / (
        df_feat["tenure"] * df_feat["MonthlyCharges"] + 1e-5
    )

    # 2. ServiceBreadth: Count of active auxiliary services (0 to 6)
    auxiliary_services = [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]
    df_feat["ServiceBreadth"] = (
        df_feat[auxiliary_services].apply(lambda row: (row == "Yes").sum(), axis=1)
    )

    # 3. HighRiskCombo: Triad uncovered in EDA (Month-to-month + Fiber Optic + No Tech Support)
    df_feat["HighRiskCombo"] = (
        (df_feat["Contract"] == "Month-to-month")
        & (df_feat["InternetService"] == "Fiber optic")
        & (df_feat["TechSupport"] == "No")
    ).astype(int)

    # 4. ContractRiskScore: Ordinal vulnerability index
    contract_risk_map = {"Month-to-month": 2, "One year": 1, "Two year": 0}
    df_feat["ContractRiskScore"] = df_feat["Contract"].map(contract_risk_map).fillna(2)

    return df_feat


X_train_engineered = engineer_saas_features(X_train)
X_test_engineered = engineer_saas_features(X_test)

print("Engineered features added successfully.")
print(X_train_engineered[["ChargeRatio", "ServiceBreadth", "HighRiskCombo", "ContractRiskScore"]].head(3))
print(f"\nColumns: hasNull Values: \n{X_train_engineered[["ChargeRatio", "ServiceBreadth", "HighRiskCombo", "ContractRiskScore"]].isna().any()}")


Engineered features added successfully.
      ChargeRatio  ServiceBreadth  HighRiskCombo  ContractRiskScore
3738     0.988182               3              0                  2
3151     1.022237               1              1                  2
4860     1.119890               3              0                  0

Columns: hasNull Values: 
ChargeRatio          False
ServiceBreadth       False
HighRiskCombo        False
ContractRiskScore    False
dtype: bool


> That output confirms your domain feature engineering worked cleanly and produced realistic values across all four engineered columns. Here is what those three customer rows tell us:

* **Row 3151 (High-Risk Profile):** Has HighRiskCombo = 1 (Month-to-month + Fiber optic + No tech support), ContractRiskScore = 2 (Month-to-month), and low product stickiness (ServiceBreadth = 1).

* **Row 3738 (Moderate-Risk Profile):** On a Month-to-month contract (ContractRiskScore = 2), but mitigated by higher product adoption (ServiceBreadth = 3) and no Fiber/Tech-support gap (HighRiskCombo = 0).

* **Row 4860 (Low-Risk / Sticky Profile):** Locked into a Two-year contract (ContractRiskScore = 0), subscribed to 3 add-on services (ServiceBreadth = 3), and shows healthy billing accrual (ChargeRatio = 1.12).

    Also, we can notice that the original DataFrame indices (3738, 3151, 4860) are preserved cleanly without index misalignment or NaN values.

---
### Constructing & Fitting the ColumnTransformer

In [3]:
# Identify explicit column groups
numeric_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ChargeRatio",
    "ServiceBreadth",
    "HighRiskCombo",
    "ContractRiskScore",
]

categorical_cols = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

# Ensure SeniorCitizen is treated as categorical string
for col in categorical_cols:
    X_train_engineered[col] = X_train_engineered[col].astype(str)
    X_test_engineered[col] = X_test_engineered[col].astype(str)

# Preprocessing Pipeline: standard scale numeric columns, one-hot encode categoricals
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="drop",
)

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train_engineered)
# Transform test data using fitted parameters
X_test_processed = preprocessor.transform(X_test_engineered)

# Extract generated one-hot feature names
cat_encoded_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_encoded_names

print(f"Total processed feature dimension: {len(all_feature_names)}")
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

Total processed feature dimension: 34
X_train_processed shape: (5634, 34)
X_test_processed shape: (1409, 34)


---
### Persisting Processed Partitions & Serializing Preprocessor

In [4]:
import os

# Ensure target directories exist
os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# 1. Save serialized Scikit-Learn transformer
joblib.dump(preprocessor, "../models/preprocessor.joblib")
print("Saved preprocessor to: ../models/preprocessor.joblib")

# 2. Save feature names for downstream SHAP alignment
with open("../models/feature_names.json", "w") as f:
    json.dump(all_feature_names, f, indent=2)
print("Saved feature names to: ../models/feature_names.json")

# 3. Save processed arrays and targets for model training
np.save("../data/processed/X_train.npy", X_train_processed)
np.save("../data/processed/X_test.npy", X_test_processed)
np.save("../data/processed/y_train.npy", y_train.to_numpy())
np.save("../data/processed/y_test.npy", y_test.to_numpy())
print("Saved processed numpy arrays into: ../data/processed/")

Saved preprocessor to: ../models/preprocessor.joblib
Saved feature names to: ../models/feature_names.json
Saved processed numpy arrays into: ../data/processed/
